# Notebook 05 : DÉMO, voir le détecteur en action sur de vrais liens
Collez une URL, le notebook affiche le **score** du détecteur, sa **décision** (phishing ou légitime), puis lance l'**attaque** et montre
l'URL modifiée, son nouveau score et la nouvelle décision, avec le détecteur normal et le détecteur durci.

**Avant de commencer** : avoir exécuté `01_defenseur.ipynb` en entier (donne le dossier `artifacts/`). Utilisez le même `PROFILE` qu'au 01.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from src.common import *
from src.attack import *
import src.attack as atk

S = load_state()
DET, BEST = load_detectors()
atk.set_bounds(S.hi)
target = DET[TARGET_NAME]
try:
    hardened = load_hardened()
except FileNotFoundError:
    hardened = None
print(f"détecteur : {target.name} | seuil : {target.thr:.3f} | score >= seuil => PHISHING")

In [ ]:
def verdict(det, url):
    s = float(det.score([url])[0])
    return s, ("PHISHING (bloqué)" if s >= det.thr else "légitime (laissé passer)")

def demo(url, budget=B_REF):
    print("URL :", url)
    s, v = verdict(target, url)
    print(f"  détecteur normal : score {s:.3f} -> {v}")
    if hardened is not None:
        sh, vh = verdict(hardened, url); print(f"  détecteur durci  : score {sh:.3f} -> {vh}")
    if s < target.thr:
        print("  (le détecteur ne la bloque déjà pas : rien à attaquer)\n"); return
    r = greedy_attack(target, url, budget)
    s2, v2 = verdict(target, r["adv"])
    print(f"  ATTAQUE (budget {budget}, {r['queries']} requêtes, opérations : {r['moves'] or 'aucune'})")
    print(f"  URL modifiée : {r['adv']}")
    print(f"  détecteur normal : score {s2:.3f} -> {v2}")
    if hardened is not None:
        sh2, vh2 = verdict(hardened, r["adv"]); print(f"  détecteur durci  : score {sh2:.3f} -> {vh2}")
    print()

## 1. Exemples tirés du jeu de validation (3 phishing, 3 légitimes)

In [ ]:
rng = np.random.default_rng(1)
ph = rng.choice(np.where(S.y_val == 1)[0], 3, replace=False)
lg = rng.choice(np.where(S.y_val == 0)[0], 3, replace=False)
for i in list(ph) + list(lg):
    print("vraie étiquette :", "PHISHING" if S.y_val[i] == 1 else "légitime")
    demo(S.urls_val[i])

## 2. Votre propre URL
Remplacez le lien ci-dessous (ne visitez pas de vrais liens de phishing : on n'analyse que le texte de l'URL, rien n'est ouvert).

In [ ]:
demo("http://secure-login.paypal-verify.xyz/account/update?id=123")

## 3. Contrôle : les modifications de l'attaque ne rendent pas tout « légitime »
On applique les **mêmes** modifications à des URL légitimes : le détecteur doit continuer à les laisser passer (sinon l'attaque ne prouverait rien).

In [ ]:
legit = S.urls_val[S.y_val == 0][:200]
mod = [candidates(u)[0][0] if candidates(u) else u for u in legit]
print("légitimes jugées phishing avant modification :", int((target.score(legit) >= target.thr).sum()), "/ 200")
print("légitimes jugées phishing après modification :", int((target.score(mod) >= target.thr).sum()), "/ 200")